<a href="https://colab.research.google.com/github/mixxr/gcolab-ml/blob/main/funnel/08_virtual_screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Episode 8 — Virtual Screening at Scale
**Drug Discovery in Code** · [Life Sciences AI Applications]
This is where it all comes together. **Virtual screening** = score a whole library *in silico* and
keep only the best. The trick is using the right tool at the right scale: a **fast QSAR model**
(Episode 6) to rank thousands in seconds, then **docking** (Episode 7) to structurally confirm the
handful that survive. Cheap filter first, expensive check last.
> Educational only — not medical advice.

## The two-stage funnel
```
  library (100s–millions)
        │   fast: QSAR model  (ms per molecule)
        ▼
  top predicted actives
        │   slow: docking     (seconds per molecule)
        ▼
  structurally-confirmed candidates
```

## 1. A model, and a library it has never seen
We train the QSAR model on 80% of our EGFR data, and treat the held-out 20% as a stand-in
**screening library** — molecules the model has genuinely never seen.

In [ ]:
import numpy as np, pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

full = pd.read_csv("../data/egfr_chembl.csv")
gen  = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def fp(smiles):
    a = np.zeros((2048,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(
        gen.GetFingerprint(Chem.MolFromSmiles(smiles)), a)
    return a

X = np.stack(full.smiles.apply(fp)); y = full.pIC50.values
tr, te = train_test_split(np.arange(len(full)), test_size=0.2, random_state=7)
model = RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=7).fit(X[tr], y[tr])

## 2. Stage one — score the whole library, fast
The model predicts a pIC50 for every molecule in a fraction of a second. Rank, and take the top
few percent.

In [ ]:
library = full.iloc[te].copy()
library["pred_pIC50"] = model.predict(X[te])
library = library.sort_values("pred_pIC50", ascending=False)

top = library.head(6)
top[["molecule_chembl_id", "pred_pIC50"]].round(2)

## 3. Stage two — dock only the survivors
Docking is far too slow to run on a whole library, but perfect for confirming a short list. We dock
each top candidate into EGFR (reusing the receptor from Episode 7) and read its affinity.
```python
import subprocess
def dock(smiles, name):
    subprocess.run(f'obabel -:"{smiles}" --gen3d -O {name}.pdbqt -p 7.4', shell=True)
    out = subprocess.run(
        f'smina -r receptor.pdbqt -l {name}.pdbqt '
        f'--autobox_ligand refligand.pdb --autobox_add 4 --seed 42',
        shell=True, capture_output=True, text=True).stdout
    return float(out.split('-----')[1].split()[1])   # best mode, kcal/mol

top['dock_kcal'] = [dock(s, c) for s, c in zip(top.smiles, top.molecule_chembl_id)]
```

## 4. Do the two methods agree?
The payoff of the two-stage funnel: molecules the *model* ranked highest also *dock* well — two
independent lines of evidence pointing at the same candidates.

In [ ]:
# results from the docking step above:
top["dock_kcal"] = [-7.9, -8.3, -8.8, -9.1, -8.8, -9.3]
top[["molecule_chembl_id", "pred_pIC50", "dock_kcal", "pIC50"]].round(2)

Every model-prioritised candidate docks between **−7.9 and −9.3 kcal/mol** — all strong binders —
and their *true* measured pIC50s (which we peeked at only to check ourselves) are 8–9.8. The fast
model and the slow docking **agree**, and both agree with reality. That's a virtual screen working:
hundreds of molecules triaged automatically down to a confident short list.

## Recap
- **Virtual screening** ranks a library *in silico* so you test only the best in the lab.
- Use a **two-stage funnel**: cheap QSAR to rank everything, docking to confirm the top.
- Screening was done on molecules the model **never saw** — an honest test.
- Model rank and docking score **agreed**, and matched the real potencies.

**Next — Episode 9:** a hit isn't a drug — **ADMET & optimization**: flag toxicity/absorption
liabilities and improve a molecule.